In [1]:
def grep(pattern):
   print("start grep for", pattern)
   while True:
       s = yield
       if pattern in s:
           print("found!", s)
       else:
           print("no %s in %s" % (pattern, s))

In [2]:
gr = grep("python")

In [3]:
next(gr)

start grep for python


In [4]:
gr.send("dwkjdwkj")

no python in dwkjdwkj


In [5]:
gr.send("python")

found! python


In [7]:
next(gr)

TypeError: argument of type 'NoneType' is not a container or iterable

In [8]:
gr.send("dwkjdwkj")

StopIteration: 

In [9]:
gr = grep("python")
gr.send(None)

start grep for python


In [10]:
gr.throw(ValueError("wrong"))

ValueError: wrong

In [11]:
def stepped_counter(start, stop, step):
    while start < stop:
        step_cand = yield start
        if isinstance(step_cand, int):
            step = step_cand

        start += step


stepped = stepped_counter(-5, 10, 1)

assert next(stepped) == -5
assert next(stepped) == -4
assert stepped.send(3) == -1
assert next(stepped) == 2
assert next(stepped) == 5
assert stepped.send(-5) == 0
assert next(stepped) == -5
assert stepped.send(7) == 2
assert next(stepped) == 9
print("ok")

next(stepped)

ok


StopIteration: 

## async

In [12]:
import asyncio
import time
import aiohttp

In [13]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    
    await say_after(1, "hello")
    await say_after(2, "world")

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 3.002717971801758


In [ ]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    
    hello = say_after(1, "hello")
    await say_after(2, "world")
    await hello

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

In [14]:
coro = say_after(2, "data")

In [15]:
await coro

data


In [16]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    
    await asyncio.create_task(say_after(1, "hello"))
    await asyncio.create_task(say_after(2, "world"))

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 3.003148078918457


In [18]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()
    
    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await task1
    await task2

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 2.00140380859375


In [19]:
async def say_after(delay, what):
    await asyncio.sleep(delay)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 2.002042770385742


In [21]:
async def say_after(delay, what):
    for _ in range(10):
        time.sleep(delay / 10)
        await asyncio.sleep(0)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 3.0759119987487793


In [29]:
async def say_after(delay, what):
    await asyncio.to_thread(time.sleep, delay)
    print(what)


async def main():
    t1 = time.time()

    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)

    t2 = time.time()
    print(f"Total time:", t2 - t1)


await main()

hello
world
Total time: 2.007932186126709


In [24]:
say = asyncio.create_task(say_after(10, "hello"))
await asyncio.sleep(2)
say.cancel()

True

In [27]:
say.cancelled()

True

In [30]:
from functools import wraps

In [31]:
def async_timeit(coro):
    @wraps(coro)
    async def inner(*args, **kwargs):
        t1 = time.time()
        try:
            return await coro(*args, **kwargs)
        finally:
            t2 = time.time()
            print(f"Time {coro.__name__}:", t2 - t1)

    return inner

In [32]:
async def say_after(delay, what):
    await asyncio.to_thread(time.sleep, delay)
    print(what)


@async_timeit
async def main():
    task1 = asyncio.create_task(say_after(1, "hello"))
    task2 = asyncio.create_task(say_after(2, "world"))

    await asyncio.gather(task1, task2)


await main()

hello
world
Time main: 2.0022029876708984


In [34]:
main.__name__

'main'

In [37]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
URLS = [URL] * 20


async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as resp:
            return (resp.status, len(await resp.text()))


await fetch_url(URL)

(200, 554202)

In [41]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 10
BATCH_SIZE = N // WORKERS


async def fetch_url(url):
    async with aiohttp.ClientSession() as session:
        async with session.get(url) as resp:
            return (resp.status, len(await resp.text()))


@async_timeit
async def fetch_batch_urls(urls, name):
    print(f"started -- {name}")
    resps = []

    for url in urls:
        resps.append(await fetch_url(url))

    print(f"finished -- {name}")
    return resps


@async_timeit
async def run():
    coros = [
        fetch_batch_urls(
            URLS[i * BATCH_SIZE : (i + 1) * BATCH_SIZE],
            f"fetcher_{i}"
        )
        for i in range(WORKERS)
    ]
    total = await asyncio.gather(*coros)
    print("total_resps", total)


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
started -- fetcher_5
started -- fetcher_6
started -- fetcher_7
started -- fetcher_8
started -- fetcher_9
finished -- fetcher_9
Time fetch_batch_urls: 2.832219123840332
finished -- fetcher_0
Time fetch_batch_urls: 3.043153762817383
finished -- fetcher_2
Time fetch_batch_urls: 3.053622007369995
finished -- fetcher_1
Time fetch_batch_urls: 3.065073013305664
finished -- fetcher_5
Time fetch_batch_urls: 3.1016337871551514
finished -- fetcher_7
Time fetch_batch_urls: 3.2008678913116455
finished -- fetcher_3
Time fetch_batch_urls: 3.2154788970947266
finished -- fetcher_4
Time fetch_batch_urls: 3.2155299186706543
finished -- fetcher_8
Time fetch_batch_urls: 3.3221607208251953
finished -- fetcher_6
Time fetch_batch_urls: 3.4023690223693848
total_resps [[(200, 554202), (200, 554202), (200, 554202), (200, 554202), (200, 554202)], [(200, 554202), (200, 554202), (200, 554202), (200, 554202), (20

In [44]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 10
BATCH_SIZE = N // WORKERS


async def fetch_url(url, session, sem):
    async with sem:
        async with session.get(url) as resp:
            return (resp.status, len(await resp.text()))


@async_timeit
async def fetch_batch_urls(urls, session, sem, name):
    print(f"started -- {name}")
    resps = []

    for url in urls:
        resps.append(await fetch_url(url, session, sem))

    print(f"finished -- {name}")
    return resps


@async_timeit
async def run():
    sem = asyncio.Semaphore(2)
    
    async with aiohttp.ClientSession() as session:
        coros = [
            fetch_batch_urls(
                URLS[i * BATCH_SIZE : (i + 1) * BATCH_SIZE],
                session,
                sem,
                f"fetcher_{i}",
            )
            for i in range(WORKERS)
        ]
        total = await asyncio.gather(*coros)

    print("total_resps", total)


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
started -- fetcher_5
started -- fetcher_6
started -- fetcher_7
started -- fetcher_8
started -- fetcher_9
finished -- fetcher_0
Time fetch_batch_urls: 2.264069080352783
finished -- fetcher_2
Time fetch_batch_urls: 2.3532228469848633
finished -- fetcher_1
Time fetch_batch_urls: 2.3856539726257324
finished -- fetcher_3
Time fetch_batch_urls: 2.455683708190918
finished -- fetcher_5
Time fetch_batch_urls: 2.504899024963379
finished -- fetcher_4
Time fetch_batch_urls: 2.540769100189209
finished -- fetcher_6
Time fetch_batch_urls: 2.6267948150634766
finished -- fetcher_8
Time fetch_batch_urls: 2.628246784210205
finished -- fetcher_9
Time fetch_batch_urls: 2.7612338066101074
finished -- fetcher_7
Time fetch_batch_urls: 2.768145799636841
total_resps [[(200, 554202), (200, 554202), (200, 554202), (200, 554202), (200, 554202)], [(200, 554202), (200, 554202), (200, 554202), (200, 554202), (200,

In [47]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 10


async def fetch_url(url, session):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()))


@async_timeit
async def worker_fetch_urls(que, session, name):
    print(f"started -- {name}")
    resps = []
    cnt = 0

    while True:
        url = await que.get()
        if url is None:
            await que.put(url)
            break

        resps.append(await fetch_url(url, session))

    print(f"finished -- {name}, {cnt=}")
    return resps


@async_timeit
async def run():
    que = asyncio.Queue()
    for url in URLS:
        await que.put(url)
    await que.put(None)
    
    async with aiohttp.ClientSession() as session:
        coros = [
            worker_fetch_urls(
                que,
                session,
                f"fetcher_{i}",
            )
            for i in range(WORKERS)
        ]
        total = await asyncio.gather(*coros)

    print("total_resps", total)


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
started -- fetcher_5
started -- fetcher_6
started -- fetcher_7
started -- fetcher_8
started -- fetcher_9
finished -- fetcher_2, cnt=0
Time worker_fetch_urls: 2.317279100418091
finished -- fetcher_7, cnt=0
Time worker_fetch_urls: 2.3250019550323486
finished -- fetcher_3, cnt=0
Time worker_fetch_urls: 2.3717498779296875
finished -- fetcher_5, cnt=0
Time worker_fetch_urls: 2.379171133041382
finished -- fetcher_1, cnt=0
Time worker_fetch_urls: 2.402480125427246
finished -- fetcher_0, cnt=0
Time worker_fetch_urls: 2.4189960956573486
finished -- fetcher_9, cnt=0
Time worker_fetch_urls: 2.428913116455078
finished -- fetcher_4, cnt=0
Time worker_fetch_urls: 2.496349811553955
finished -- fetcher_6, cnt=0
Time worker_fetch_urls: 2.4986770153045654
finished -- fetcher_8, cnt=0
Time worker_fetch_urls: 2.514681100845337
total_resps [[(200, 554202), (200, 554202), (200, 554202), (200, 554202), (2

In [49]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 10


async def fetch_url(url, session):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()))


@async_timeit
async def worker_fetch_urls(que, session, name):
    print(f"started -- {name}")
    resps = []
    cnt = 0

    while True:
        url = await que.get()

        resps.append(await fetch_url(url, session))
        cnt += 1
        que.task_done()

    print(f"finished -- {name}, {cnt=}")
    return resps


@async_timeit
async def run():
    que = asyncio.Queue()
    for url in URLS:
        await que.put(url)

    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(
                worker_fetch_urls(que, session, f"fetcher_{i}")
            )
            for i in range(WORKERS)
        ]
        await que.join()

        for worker in workers:
            worker.cancel()

    print("total_resps")


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
started -- fetcher_5
started -- fetcher_6
started -- fetcher_7
started -- fetcher_8
started -- fetcher_9
Time worker_fetch_urls: 3.955435037612915
Time worker_fetch_urls: 3.9552037715911865
Time worker_fetch_urls: 3.9551172256469727
Time worker_fetch_urls: 3.9550180435180664
Time worker_fetch_urls: 3.9549403190612793
Time worker_fetch_urls: 3.95489501953125
Time worker_fetch_urls: 3.9548990726470947
Time worker_fetch_urls: 3.9548890590667725
Time worker_fetch_urls: 3.9548449516296387
Time worker_fetch_urls: 3.954793930053711
total_resps
Time run: 3.957489252090454
